This script clean SMILES strings using RDKit which are extracted from existing CSV (with columns like molecule_chembl_id, canonical_smiles, IC50_nM, pIC50, etc.). This script performs:

1. Salt removal

2. Largest fragment selection

3. Charge neutralization

4. Canonicalization for uniformity

In [1]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive


In [2]:
!pip install rdkit

In [3]:
import pandas as pd
from rdkit import Chem
from rdkit.Chem.MolStandardize import rdMolStandardize
from rdkit.Chem import SaltRemover
import os

In [15]:
# Load  dataset
input_path = '/content/drive/MyDrive/Colab Notebooks/ESOFT-MSC/thesis-project/data/processed-data/activity-labeled-data/'
file_name = 'mao-b_cleaned_labeled.csv'
df = pd.read_csv(input_path + file_name)
df.head()

,molecule_chembl_id,canonical_smiles,IC50_nM,pIC50,target_chembl_id,bioactivity_class
0,CHEMBL350093,N#CCCN1CC(=O)OC(c2ccc(OCc3ccccc3)cc2)=N1,18.0,7.744727,CHEMBL2039,active
1,CHEMBL161907,O=c1c(=O)c2ccc(OCCCC(F)(F)F)cc2c1=O,9.0,8.045757,CHEMBL2039,active
2,CHEMBL17079,N#CCCn1nc(-c2ccc(OCc3ccccc3)cc2)oc1=S,4.4,8.356547,CHEMBL2039,active
3,CHEMBL160347,COc1cc(Br)c2oc(C3CCNCC3)cc2c1,23400.0,4.630784,CHEMBL2039,inactive
4,CHEMBL347197,CC/N=C1/CCc2c1n(C)c1ccccc21,77600.0,4.110138,CHEMBL2039,inactive


In [5]:
# RDKit standardizer components


# Set up the standardizer components
remover = SaltRemover.SaltRemover()                           # removes common salts
fragment_remover = rdMolStandardize.LargestFragmentChooser()  # keeps only largest fragment
uncharger = rdMolStandardize.Uncharger()                      # removes formal charges

def clean_smiles(smiles):
    try:
        mol = Chem.MolFromSmiles(smiles)
        if mol is None:
            return None

        # Clean up using RDKit tools
        mol = remover.StripMol(mol, dontRemoveEverything=True)  # Remove salts
        mol = fragment_remover.choose(mol)                      # Retain largest fragment
        mol = uncharger.uncharge(mol)                           # Neutralize

        # Optional: Final conversion to canonical SMILES
        return Chem.MolToSmiles(mol, isomericSmiles=False)

    except Exception as e:
        print(f"Error cleaning SMILES: {e}")
        return None




In [6]:
# Apply cleaning to 'canonical_smiles' and store in new column
df['cleaned_smiles'] = df['canonical_smiles'].apply(clean_smiles)

# Drop rows where cleaning failed
df = df.dropna(subset=['cleaned_smiles']).reset_index(drop=True)

# Optionally replace original column
df['canonical_smiles'] = df['cleaned_smiles']
df = df.drop(columns=['cleaned_smiles'])

Streaming output truncated to the last 5000 lines.
[13:48:53] Running Uncharger
[13:48:53] Running LargestFragmentChooser
[13:48:53] Running Uncharger
[13:48:54] Running LargestFragmentChooser
[13:48:54] Running Uncharger
[13:48:54] Running LargestFragmentChooser
[13:48:54] Running Uncharger
[13:48:54] Running LargestFragmentChooser
[13:48:54] Running Uncharger
[13:48:54] Running LargestFragmentChooser
[13:48:54] Running Uncharger
[13:48:54] Running LargestFragmentChooser
[13:48:54] Running Uncharger
[13:48:54] Running LargestFragmentChooser
[13:48:54] Running Uncharger
[13:48:54] Running LargestFragmentChooser
[13:48:54] Running Uncharger
[13:48:54] Running LargestFragmentChooser
[13:48:54] Running Uncharger
[13:48:54] Running LargestFragmentChooser
[13:48:54] Running Uncharger
[13:48:54] Running LargestFragmentChooser
[13:48:54] Running Uncharger
[13:48:54] Running LargestFragmentChooser
[13:48:54] Running Uncharger
[13:48:54] Running LargestFragmentChooser
[13:48:54] Running Uncharg

In [7]:
df.head()

,molecule_chembl_id,canonical_smiles,IC50_nM,pIC50,target_chembl_id,bioactivity_class
0,CHEMBL350093,N#CCCN1CC(=O)OC(c2ccc(OCc3ccccc3)cc2)=N1,18.0,7.744727,CHEMBL2039,active
1,CHEMBL161907,O=c1c(=O)c2ccc(OCCCC(F)(F)F)cc2c1=O,9.0,8.045757,CHEMBL2039,active
2,CHEMBL17079,N#CCCn1nc(-c2ccc(OCc3ccccc3)cc2)oc1=S,4.4,8.356547,CHEMBL2039,active
3,CHEMBL160347,COc1cc(Br)c2oc(C3CCNCC3)cc2c1,23400.0,4.630784,CHEMBL2039,inactive
4,CHEMBL347197,CCN=C1CCc2c1n(C)c1ccccc21,77600.0,4.110138,CHEMBL2039,inactive


In [8]:
# Save cleaned data
output_file = input_path + file_name.replace(".csv", "_smiles_cleaned.csv")
df.to_csv(output_file, index=False)

print("SMILES cleaning complete. Cleaned file saved to:")
print(output_file)

SMILES cleaning complete. Cleaned file saved to:
/content/drive/MyDrive/Colab Notebooks/ESOFT-MSC/thesis-project/data/processed-data/activity-labeled-data/mao-b_cleaned_labeled_smiles_cleaned.csv
